# Customer Segmentation: Unsupervised Learning Case Study

A comprehensive analysis of customer data using multiple clustering techniques — K-Means, Hierarchical Clustering, DBSCAN, and Gaussian Mixture Models — to identify actionable customer segments for targeted marketing strategies.


## Problem Statement

Businesses need to understand their customer base to deliver personalized marketing, improve retention, and maximize revenue. The goal of this project is to segment customers into distinct groups based on their purchasing behavior and demographic attributes. Each segment will be profiled and given specific marketing recommendations.

**Features used for segmentation:**
- Age
- Annual Income
- Spending Score (1–100)
- Purchase Frequency (per month)
- Recency (days since last purchase)
- Loyalty Score (1–5)
- Average Order Value ($)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
from math import pi
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 10)
print('All libraries loaded successfully.')

## 1. Generate Synthetic Customer Data

We simulate four underlying customer segments with realistic parameter ranges, then add a small amount of Gaussian noise. The true segment labels are kept for validating clustering performance later.

In [ ]:
np.random.seed(42)
n_customers = 1000

segments = {
    'Young Explorers': {
        'age': (18, 30), 'income': (20000, 50000),
        'spending_score': (60, 100), 'purchase_frequency': (1, 5),
        'recency': (1, 15), 'loyalty_score': (1, 3),
        'avg_order_value': (20, 80)
    },
    'Established Professionals': {
        'age': (30, 50), 'income': (60000, 120000),
        'spending_score': (40, 80), 'purchase_frequency': (3, 8),
        'recency': (3, 30), 'loyalty_score': (3, 5),
        'avg_order_value': (50, 150)
    },
    'Budget Conscious': {
        'age': (25, 60), 'income': (15000, 35000),
        'spending_score': (10, 40), 'purchase_frequency': (1, 3),
        'recency': (10, 60), 'loyalty_score': (1, 2),
        'avg_order_value': (10, 40)
    },
    'Premium Loyalists': {
        'age': (40, 70), 'income': (80000, 200000),
        'spending_score': (70, 100), 'purchase_frequency': (5, 12),
        'recency': (1, 10), 'loyalty_score': (4, 5),
        'avg_order_value': (100, 300)
    }
}

data_frames = []
feature_cols = ['age', 'income', 'spending_score', 'purchase_frequency',
                'recency', 'loyalty_score', 'avg_order_value']

for label, params in segments.items():
    n = n_customers // len(segments)
    df_seg = pd.DataFrame({
        'age': np.random.uniform(*params['age'], n),
        'income': np.random.uniform(*params['income'], n),
        'spending_score': np.random.uniform(*params['spending_score'], n),
        'purchase_frequency': np.random.uniform(*params['purchase_frequency'], n),
        'recency': np.random.uniform(*params['recency'], n),
        'loyalty_score': np.random.uniform(*params['loyalty_score'], n),
        'avg_order_value': np.random.uniform(*params['avg_order_value'], n),
        'true_segment': label
    })
    data_frames.append(df_seg)

df = pd.concat(data_frames, ignore_index=True)

for col in feature_cols:
    noise = np.random.normal(0, df[col].std() * 0.1, len(df))
    df[col] = df[col] + noise
    df[col] = df[col].clip(lower=0)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Generated {len(df)} customer records with {len(segments)} true segments.')
df.head()

## 2. Exploratory Data Analysis

We examine distributions, correlations, and pairwise relationships to understand the structure of our customer data before clustering.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.flat, feature_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}', fontsize=12)
fig.delaxes(axes[2, 2])
plt.suptitle('Visualization 1: Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[feature_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Visualization 2: Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df, vars=feature_cols, hue='true_segment',
             diag_kind='kde', palette='Set2', height=1.8)
plt.suptitle('Visualization 3: Pairplot of Customer Features by True Segment',
             y=1.02, fontsize=14)
plt.show()

## 3. Feature Scaling

Clustering algorithms are distance-sensitive. We standardize all numerical features to zero mean and unit variance so that no single feature dominates the distance calculation.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])
print(f'Scaled feature matrix shape: {X_scaled.shape}')
print(f'Mean after scaling: {X_scaled.mean(axis=0).round(4)}')
print(f'Std after scaling:  {X_scaled.std(axis=0).round(4)}')

## 4. K-Means Clustering

We determine the optimal number of clusters via the Elbow Method and Silhouette Score, then visualize the silhouette coefficients for the chosen k.

In [ ]:
inertias, silhouettes = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].axvline(x=4, color='r', linestyle='--', alpha=0.7, label='k=4')
axes[0].set(title='Elbow Method', xlabel='Number of clusters (k)', ylabel='Inertia')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'go-')
axes[1].axvline(x=4, color='r', linestyle='--', alpha=0.7, label='k=4')
axes[1].set(title='Silhouette Score', xlabel='Number of clusters (k)', ylabel='Score')
axes[1].legend()

plt.suptitle('Visualization 4: Optimal k Selection — Elbow + Silhouette', fontsize=14)
plt.tight_layout()
plt.show()

best_k = 4
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km.fit_predict(X_scaled)
print(f'Chosen k = {best_k} with Silhouette Score = {silhouettes[best_k-2]:.4f}')

In [ ]:
silhouette_vals = silhouette_samples(X_scaled, km_labels)
y_lower = 10

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(best_k):
    vals = silhouette_vals[km_labels == i]
    vals.sort()
    size = len(vals)
    y_upper = y_lower + size
    color = plt.cm.viridis(i / best_k)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                     alpha=0.7, color=color, label=f'Cluster {i}')
    y_lower = y_upper + 10

avg_score = silhouette_score(X_scaled, km_labels)
ax.axvline(x=avg_score, color='red', linestyle='--', label=f'Avg = {avg_score:.3f}')
ax.set(title='Visualization 5: Silhouette Plot for K-Means (k=4)',
       xlabel='Silhouette Coefficient', ylabel='Cluster')
ax.legend()
plt.show()

## 5. Hierarchical Clustering

We apply Ward's linkage and inspect the dendrogram to visualize the hierarchical relationships between customers.

In [ ]:
Z = linkage(X_scaled, method='ward')
plt.figure(figsize=(14, 7))
dendrogram(Z, truncate_mode='level', p=5, color_threshold=0.7 * max(Z[:, 2]))
plt.title('Visualization 6: Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.xlabel('Customer Index (truncated)')
plt.ylabel('Ward Distance')
plt.axhline(y=0.7 * max(Z[:, 2]), color='r', linestyle='--', label='Cut threshold')
plt.legend()
plt.show()

hier = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
hier_labels = hier.fit_predict(X_scaled)
print(f'Hierarchical ARI vs K-Means: {adjusted_rand_score(km_labels, hier_labels):.4f}')

## 6. DBSCAN — Density-Based Clustering

DBSCAN finds clusters of arbitrary shape and identifies noise points (outliers). This is useful for detecting customers with unusual behavior.

In [ ]:
dbscan = DBSCAN(eps=0.6, min_samples=10)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)
noise_pct = n_noise / len(db_labels) * 100

print(f'DBSCAN found {n_clusters_db} clusters and {n_noise} noise points ({noise_pct:.1f}%)')
print(f'DBSCAN ARI vs K-Means: {adjusted_rand_score(km_labels, db_labels):.4f}')

## 7. Gaussian Mixture Models — Soft Clustering

GMM assigns each customer a probability of belonging to each cluster, enabling soft (probabilistic) segmentation.

In [ ]:
gmm = GaussianMixture(n_components=best_k, random_state=42, n_init=10)
gmm_labels = gmm.fit_predict(X_scaled)
gmm_probs = gmm.predict_proba(X_scaled)

print(f'GMM log-likelihood: {gmm.score(X_scaled):.2f}')
print(f'GMM ARI vs K-Means: {adjusted_rand_score(km_labels, gmm_labels):.4f}')
print(f'GMM ARI vs True Labels: {adjusted_rand_score(df["true_segment"], gmm_labels):.4f}')

## 8. Visualizing Segments in 2D (PCA Projection)

We project the 7-dimensional data onto two principal components to visualize how each clustering algorithm partitions the customer space.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}')
print(f'Total explained variance: {pca.explained_variance_ratio_.sum():.2%}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plot_data = [
    (km_labels, 'K-Means', 'viridis'),
    (db_labels, 'DBSCAN', 'tab10'),
    (gmm_labels, 'GMM', 'plasma')
]

for ax, (labels, title, cmap) in zip(axes, plot_data):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap=cmap,
                         alpha=0.7, edgecolors='k', s=50)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.colorbar(scatter, ax=ax, shrink=0.8)

plt.suptitle("Visualization 7: Customer Segments in PCA-Reduced Space", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Segment Profiling

We assign K-Means segments back to the dataframe and compute the mean characteristics of each cluster. A radar chart provides a compact visual summary of the segments.

In [ ]:
df['kmeans_segment'] = km_labels
profile = df.groupby('kmeans_segment')[feature_cols].mean()
profile_std = df.groupby('kmeans_segment')[feature_cols].std()

print('Segment Profiles (mean values):')
display(profile.round(2))
print('\nSegment Profiles (std values):')
display(profile_std.round(2))

In [ ]:
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())
categories = feature_cols
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for idx in profile_norm.index:
    values = profile_norm.loc[idx].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Segment {idx}')
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_title('Visualization 8: Customer Segments Radar Chart',
             pad=30, size=14, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.show()

## 10. Cluster Characteristics Comparison & DBSCAN vs K-Means

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flat, feature_cols):
    for seg in range(best_k):
        data = df[df['kmeans_segment'] == seg][col]
        ax.hist(data, alpha=0.5, label=f'Segment {seg}', bins=20)
    ax.set_title(col, fontsize=12)
    ax.legend(fontsize=8)
fig.delaxes(axes[2, 2])
plt.suptitle('Visualization 9: Feature Distributions by Segment', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='viridis',
                alpha=0.7, edgecolors='k', s=50)
axes[0].set_title(f'K-Means (k={best_k})', fontsize=12)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels, cmap='tab10',
                alpha=0.7, edgecolors='k', s=50)
axes[1].set_title(f'DBSCAN ({n_clusters_db} clusters, {n_noise} noise points)', fontsize=12)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.suptitle('Visualization 10: DBSCAN vs K-Means Comparison',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 11. Stability Analysis

We evaluate how consistent K-Means segment assignments are across different random initializations using the Adjusted Rand Index (ARI).

In [ ]:
n_iterations = 20
ari_scores = []
km_base = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X_scaled)

for i in range(n_iterations):
    km_iter = KMeans(n_clusters=best_k, random_state=i, n_init=10).fit_predict(X_scaled)
    ari_scores.append(adjusted_rand_score(km_base, km_iter))

plt.figure(figsize=(10, 5))
plt.plot(range(n_iterations), ari_scores, 'bo-')
plt.axhline(y=np.mean(ari_scores), color='r', linestyle='--',
            label=f'Mean ARI: {np.mean(ari_scores):.3f}')
plt.fill_between(range(n_iterations),
                 np.mean(ari_scores) - np.std(ari_scores),
                 np.mean(ari_scores) + np.std(ari_scores),
                 alpha=0.2, color='red')
plt.title('K-Means Stability Across Random Initializations')
plt.xlabel('Iteration')
plt.ylabel('Adjusted Rand Index (vs Base)')
plt.legend()
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()

print(f'Stability assessment: mean ARI = {np.mean(ari_scores):.3f} +/- {np.std(ari_scores):.3f}')
if np.mean(ari_scores) > 0.95:
    print('Segments are highly stable across initializations.')
elif np.mean(ari_scores) > 0.8:
    print('Segments are reasonably stable.')
else:
    print('Segments show instability — consider increasing n_init or reducing k.')

## 12. Marketing Recommendations

Each segment's profile drives targeted marketing strategies.

In [ ]:
def recommend_strategy(seg_id, row):
    print(f"{'='*60}")
    print(f"  SEGMENT {seg_id}")
    print(f"{'='*60}")
    print(f"  Age range:           {row['age']:.0f} years (mean)")
    print(f"  Annual Income:       ${row['income']:,.0f}")
    print(f"  Spending Score:      {row['spending_score']:.1f}/100")
    print(f"  Purchase Frequency:  {row['purchase_frequency']:.1f}/month")
    print(f"  Recency:             {row['recency']:.0f} days")
    print(f"  Loyalty Score:       {row['loyalty_score']:.1f}/5")
    print(f"  Avg Order Value:     ${row['avg_order_value']:.0f}")
    print()

    if row['spending_score'] > 70 and row['income'] > 80000:
        print("  >> Strategy: Premium Retention")
        print("  >> VIP loyalty program, early access to new products,")
        print("      personal shopping assistants, exclusive events.")
    elif row['spending_score'] > 50 and row['income'] > 50000:
        print("  >> Strategy: Growth & Upsell")
        print("  >> Targeted email campaigns, product bundles,")
        print("      limited-time offers, referral incentives.")
    elif row['recency'] > 30:
        print("  >> Strategy: Re-engagement / Win-back")
        print("  >> Discount vouchers, 'We miss you' emails,")
        print("      free shipping on next order.")
    else:
        print("  >> Strategy: Value & Loyalty")
        print("  >> Loyalty points program, budget-friendly bundles,")
        print("      newsletters with tips and deals.")

for seg_id in range(best_k):
    recommend_strategy(seg_id, profile.loc[seg_id])

## Conclusion

We successfully segmented synthetic customer data using four clustering approaches. K-Means with k=4 produced interpretable, stable segments. Hierarchical clustering confirmed the grouping structure via dendrogram. DBSCAN identified noise points (outliers) that merit individual attention. GMM provided probabilistic memberships for soft segmentation.

**Key takeaways:**
- StandardScaler is essential for distance-based clustering.
- Multiple algorithms should be compared — no single method is universally best.
- Segment profiling bridges technical clustering output to actionable business strategy.